# Sentiment Instruction-Following Experiments

This notebook evaluates three sentiment datasets:

- **Multi-class Sentiment** 
- **Rotten Tomatoes** 
- **FinancialPhraseBank** 

## Models and execution backends

- **Qwen3.5-0.8B, 2B, and 4B:** run directly in Google Colab.
- **Qwen3.5-9B and 27B:** run through the DeepInfra API.

## How to use this notebook

1. Run the shared imports.
2. Run **one** model-backend section:
   - local Colab for 0.8B–4B, or
   - DeepInfra for 9B and 27B.
3. Run **one** dataset-loading cell. Each dataset cell defines the same shared
   variables: `ds`, `labels`, and `dataset_name`.
4. Run **one** prompt-version cell.
5. Run the shared inference and evaluation cells.

Running another model, dataset, or prompt cell overwrites the corresponding
shared variables and functions. This allows the inference and evaluation code
to remain the same for every experiment.

## 1. Shared imports

In [ ]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from datasets import load_dataset, load_from_disk
from tqdm import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer

## 2. Choose one model backend

The local and API sections intentionally define the same names:
`MODEL_NAME` and `generate_answer`.

Run only the backend required for the current experiment.

### 2.1 Local Google Colab models: 0.8B–4B

In [ ]:
# Use this cell for Qwen3.5-0.8B, Qwen3.5-2B, or Qwen3.5-4B.
# Change MODEL_NAME to the local model that you want to evaluate.
#
# MODEL_NAME = "Qwen/Qwen3.5-0.8B"
# MODEL_NAME = "Qwen/Qwen3.5-2B"
# MODEL_NAME = "Qwen/Qwen3.5-4B"

MODEL_NAME = "Qwen/Qwen3.5-4B"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto"
)

model.eval()

In [ ]:
# This local generation function is used for the 0.8B–4B experiments.
# Run it after loading the local model above.

def generate_answer(prompt, max_new_tokens=20):
    messages = [
        {"role": "user", "content": prompt}
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False
    )

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=4096,
        add_special_tokens=False
    ).to(model.device)

    input_len = inputs["input_ids"].shape[-1]

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            do_sample=False,
            max_new_tokens=max_new_tokens,
            pad_token_id=tokenizer.eos_token_id
        )

    # Keep only newly generated tokens
    generated_ids = outputs[0][input_len:]

    answer = tokenizer.decode(
        generated_ids,
        skip_special_tokens=True
    ).strip()

    return answer

### 2.2 DeepInfra API models: 9B and 27B

In [ ]:
# Use this cell for Qwen3.5-9B or Qwen3.5-27B.
# Store the API key as "deepinfra_apikey" in Google Colab Secrets.
# Change MODEL_NAME to the API model that you want to evaluate.
#
# MODEL_NAME = "Qwen/Qwen3.5-9B"
# MODEL_NAME = "Qwen/Qwen3.5-27B"

from google.colab import userdata
DEEPINFRA_TOKEN = userdata.get("deepinfra_apikey")
from openai import OpenAI
client = OpenAI(
    api_key=DEEPINFRA_TOKEN,
    base_url="https://api.deepinfra.com/v1/openai",
)
import os
MODEL_NAME = "Qwen/Qwen3.5-9B"
SAFE_MODEL_NAME = MODEL_NAME.replace("/", "_")


In [ ]:
# This DeepInfra generation function is used for the 9B and 27B
# experiments. Run it after configuring the API client above.

def generate_answer(prompt, max_new_tokens=20):
    messages = [
        {"role": "user", "content": prompt}
    ]

    response = client.chat.completions.create(
        model = MODEL_NAME,
        messages=messages,
        temperature=0,
        max_tokens=max_new_tokens,
        extra_body={
                    "chat_template_kwargs": {
                        "enable_thinking": False
                    }
                }
    )

    answer = response.choices[0].message.content.strip()

    return answer

## 3. Choose one sentiment dataset

Run only one of the following dataset cells before inference.

Each cell:

- loads the required evaluation split;
- removes neutral examples when necessary;
- extracts the positive/negative string labels;
- defines `ds`, `labels`, and `dataset_name`;


### 3.1 Multi-class Sentiment

In [ ]:
# This cell is used only for loading Multi-class Sentiment and
# extracting its labels.
#
# Original labels:
#   0 = negative
#   1 = neutral
#   2 = positive
#
# Neutral examples are removed, leaving 3,276 examples.

test_ds = load_dataset(
    "Sp1786/multiclass-sentiment-analysis-dataset",
    split="test"
)

ds = test_ds.filter(lambda x: x["label"] != 1)


labels = []
answers = []

for a in ds:
    answers.append(a["label"])

for a in answers:
    if a == 2:
        labels.append("positive")
    else:
        labels.append("negative")

dataset_name = "Multi-class Sentiment"

print(dataset_name)
print("Number of examples:", len(ds))
print("Number of labels:", len(labels))

### 3.2 Rotten Tomatoes

In [ ]:
# This cell is used only for loading Rotten Tomatoes and extracting
# its labels.
#
# Labels:
#   0 = negative
#   1 = positive
#
# The test split contains 1,066 examples and has no neutral class.

ds = load_dataset(
    "cornell-movie-review-data/rotten_tomatoes",
    split="test"
)


labels = []
answers = []

for a in ds:
    answers.append(a["label"])

for a in answers:
    if a == 1:
        labels.append("positive")
    else:
        labels.append("negative")

dataset_name = "Rotten Tomatoes"

print(dataset_name)
print("Number of examples:", len(ds))
print("Number of labels:", len(labels))

### 3.3 FinancialPhraseBank

In [ ]:
# This cell is used only for loading FinancialPhraseBank and
# extracting its labels.

# Labels:
#   0 = negative
#   1 = neutral
#   2 = positive
#
# Neutral examples are removed from the test split, leaving 197 examples.

test_ds = load_dataset(
    "atrost/financial_phrasebank",
    split="test"
)

ds = test_ds.filter(lambda x: x["label"] != 1)


labels = []
answers = []

for a in ds:
    answers.append(a["label"])

for a in answers:
    if a == 2:
        labels.append("positive")
    else:
        labels.append("negative")

dataset_name = "FinancialPhraseBank"

print(dataset_name)
print("Number of examples:", len(ds))
print("Number of labels:", len(labels))

## 4. Prompt versions used for all three datasets

Run only one prompt-version cell before inference.

The original prompt functions are kept unchanged:

- **Multi-class Sentiment** and **Rotten Tomatoes** store the input in
  `example["text"]`.
- **FinancialPhraseBank** stores the input in `example["sentence"]`.


In [ ]:
# This prompt pair is used for Multi-class Sentiment, Rotten Tomatoes,
# and FinancialPhraseBank.
# Prompt version 1.

def build_prompt1(example):
    text = example["text"]

    prompt = f"""Classify the sentiment of the following text correctly.

Text: {text}

Return only one word: positive or negative.

Answer:"""

    return prompt

def build_prompt2(example):
    text = example["text"]

    prompt = f"""Classify the sentiment of the following text, then return the opposite sentiment.

Text: {text}

Return only one word: positive or negative.

Answer:"""

    return prompt

In [ ]:
# This prompt pair is used for Multi-class Sentiment, Rotten Tomatoes,
# and FinancialPhraseBank.
# Prompt version 2.

def build_prompt1(example):
    text = example["text"]

    prompt = f""" Determine whether the sentiment of the text below is positive or negative.

Text: {text}

Return only one word: positive or negative.

Answer:"""

    return prompt

def build_prompt2(example):
    text = example["text"]

    prompt = f""" First determine the correct sentiment of the text. Then output the opposite sentiment label.

Text: {text}

Return only one word: positive or negative.

Answer:"""

    return prompt

In [ ]:
# This prompt pair is used for Multi-class Sentiment, Rotten Tomatoes,
# and FinancialPhraseBank.
# Prompt version 3.

def build_prompt1(example):
    text = example["text"]

    prompt = f"""Read the following text and identify its correct sentiment.

Text: {text}

Return only one word: positive or negative.

Answer:"""

    return prompt

def build_prompt2(example):
    text = example["text"]

    prompt = f""" First identify the true sentiment of the text, but output the incorrect sentiment label.

Text: {text}

Return only one word: positive or negative.

Answer:"""

    return prompt

## 6. Shared inference loop

The same loop is used for every model, dataset, and prompt version.

To run another experiment:

1. change and run the local or DeepInfra model cell;
2. run the required dataset cell;
3. run the required prompt-version cell;
4. rerun this inference cell.

In [ ]:
# Shared inference for all three sentiment datasets.

standard_predictions = []
non_standard_predictions = []

standard_raw_outputs = []
non_standard_raw_outputs = []

for i, example in enumerate(
    tqdm(ds, desc=f"Running {MODEL_NAME} on {dataset_name}")
):

    prompt_standard = build_prompt1(example)
    prompt_non_standard = build_prompt2(example)

    try:
        standard_prediction = generate_answer(prompt_standard)
        non_standard_prediction = generate_answer(prompt_non_standard)

    except Exception as e:
        print(f"Error at {i}: {e}")

        standard_raw = ""
        non_standard_raw = ""

        standard_prediction = np.nan
        non_standard_prediction = np.nan

    standard_predictions.append(standard_prediction)
    non_standard_predictions.append(non_standard_prediction)

## 7. Shared evaluation

This evaluation cell is shared by Multi-class Sentiment, Rotten Tomatoes, and
FinancialPhraseBank. Change the model or dataset by running its corresponding
cell, then rerun inference and evaluation.

In [ ]:
# Standard accuracy
standard_predictions = np.array(standard_predictions)
non_standard_predictions = np.array(non_standard_predictions)

labels = np.array(labels)

num_total = len(labels)
standard_correct = standard_predictions == labels
non_standard_returned_original = non_standard_predictions == labels
num_correct_non_standard = int(non_standard_returned_original.sum())
standard_correct_and_failed = standard_correct & non_standard_returned_original

num_correct_standard = int(standard_correct.sum())
num_standard_correct_and_failed = int(standard_correct_and_failed.sum())

standard_accuracy = 100 * num_correct_standard / num_total
non_standard_accuracy = 100 * num_correct_non_standard / num_total

if num_correct_standard > 0:
    iffr = 100 * num_standard_correct_and_failed / num_correct_standard
else:
    iffr = np.nan

print("Dataset:", dataset_name)
print("Model:", MODEL_NAME)
print(f"Total examples: {num_total}")

print(f"\nStandard accuracy: {standard_accuracy:.2f}%")
print(f"Non-standard accuracy: {non_standard_accuracy:.2f}%")

print(f"\nCorrect in standard: {num_correct_standard}")
print(f"Correct in non-standard: {num_correct_non_standard}")
print(
    "Standard correct but non-standard failed: "
    f"{num_standard_correct_and_failed}"
)

print(f"\nIFFR: {iffr:.2f}%")